In [1]:
%%capture
!pip install kaggle

In [ ]:
import os
import shutil
import cv2
import json
import subprocess
import logging
import math
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from PIL import Image

class FrameExtractor:
    def __init__(self, input_path: str, output_path: str, kaggle_api: dict, base_dataset_name: str, progress_info: dict = {}):
        """
        Initializes the FrameExtractor.

        Args:
            input_path (str): Path to the directory containing input videos.
            output_path (str): Path to the directory to save extracted frames.
            kaggle_api (dict): Dictionary containing Kaggle API 'username' and 'key'.
            base_dataset_name (str): Base name for the datasets to be created on Kaggle.
            progress_info (dict, optional): Dictionary to control partial processing. 
                                            Defaults to {}. If empty, all videos will be processed.
                                            Example: {"start_index": 0, "end_index": 100}
        """
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.upload_dir = self.output_path / "upload"
        self.progress_info = progress_info or {}

        self.kaggle_username = kaggle_api.get("username")
        self.kaggle_key = kaggle_api.get("key")
        if not self.kaggle_username or not self.kaggle_key:
            raise ValueError("Kaggle API 'username' and 'key' must not be empty.")

        # Setup Kaggle API credentials
        kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(kaggle_json_path, "w") as f:
            json.dump({"username": self.kaggle_username, "key": self.kaggle_key}, f)
        os.chmod(kaggle_json_path, 0o600)

        self.base_dataset_name = base_dataset_name
        self.width = 0
        self.height = 0

        # Clean up and create necessary directories
        print(f"Cleaning up output directory '{self.output_path}'...")
        shutil.rmtree(self.output_path, ignore_errors=True)
        self.output_path.mkdir(parents=True, exist_ok=True)
        self.upload_dir.mkdir(exist_ok=True)

    def get_video_files(self, file_extension: str = '.mp4') -> list:
        """
        Gets a list of video files from the input path.
        Filters the list based on self.progress_info if provided.
        """
        file_list = sorted([Path(root) / f for root, _, files in os.walk(self.input_path) for f in files if f.endswith(file_extension)])
        print(f"Found a total of {len(file_list)} video files with extension '{file_extension}'.")

        if not self.progress_info:
            print("No progress info provided, processing all files.")
            self.progress_info = {
                "start_index": 0,
                "end_index": len(file_list)
            }

        start_index = self.progress_info.get("start_index", 0)
        end_index = self.progress_info.get("end_index", len(file_list))
        
        print(f"Processing files from index {start_index} to {end_index}.")
        return file_list[start_index:end_index]

    def _get_video_metadata(self, video_path: Path) -> dict:
        """Gets video metadata such as FPS and duration."""
        try:
            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration_seconds = frame_count / fps if fps > 0 else 0
            cap.release()
            return {"fps": fps, "duration": duration_seconds}
        except Exception:
            return {"fps": 0, "duration": 0}

    def _resize_or_pad(self, image, new_size):
        """Resizes or pads an image to fit the target dimensions while preserving aspect ratio."""
        h, w = image.shape[:2]
        target_w, target_h = new_size
        
        # Maintain aspect ratio
        if w / h > target_w / target_h:
            # Fit to width
            scale = target_w / w
            resized_h = int(h * scale)
            resized_image = cv2.resize(image, (target_w, resized_h))
            p_top = (target_h - resized_h) // 2
            p_bottom = target_h - resized_h - p_top
            return cv2.copyMakeBorder(resized_image, p_top, p_bottom, 0, 0, cv2.BORDER_CONSTANT, value=[0, 0, 0])
        else:
            # Fit to height
            scale = target_h / h
            resized_w = int(w * scale)
            resized_image = cv2.resize(image, (resized_w, target_h))
            p_left = (target_w - resized_w) // 2
            p_right = target_w - resized_w - p_left
            return cv2.copyMakeBorder(resized_image, 0, 0, p_left, p_right, cv2.BORDER_CONSTANT, value=[0, 0, 0])

    def _process_single_video(self, video_path: Path) -> Path:
        """Processes a single video: extracts, resizes/pads, and saves frames."""
        video_name_stem = video_path.stem.replace(' ', '_')
        frame_out_dir = self.output_path / video_name_stem
        frame_out_dir.mkdir(exist_ok=True)
        
        metadata = self._get_video_metadata(video_path)
        with open(frame_out_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=4)
            
        cap = cv2.VideoCapture(str(video_path))
        frame_index = 0
        saved_frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # Extract every 7th frame
            if frame_index % 7 == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                processed_frame = self._resize_or_pad(frame_rgb, (self.width, self.height))
                Image.fromarray(processed_frame).save(frame_out_dir / f'{saved_frame_count}.jpg')
                saved_frame_count +=1
            frame_index += 1
        cap.release()
        return frame_out_dir

    def _get_dir_size_in_gb(self, dir_path: Path) -> float:
        """Calculates the total size of a directory in GB."""
        total_size = sum(f.stat().st_size for f in dir_path.glob('**/*') if f.is_file())
        return total_size / (1024 ** 3)

    def _create_new_dataset(self, dataset_name: str, message: str):
        """Creates a new dataset on Kaggle."""
        dataset_slug = dataset_name.lower().replace(' ', '-')
        dataset_id = f"{self.kaggle_username}/{dataset_slug}"
        print(f"Creating new dataset: {dataset_id}")
        dataset_meta = {
            "title": dataset_name,
            "id": dataset_id,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(self.upload_dir / 'dataset-metadata.json', 'w') as f:
            json.dump(dataset_meta, f, indent=4)

        command = ["kaggle", "datasets", "create", "-p", str(self.upload_dir), "--dir-mode", "zip"]
        try:
            print(f"Executing command: {' '.join(command)}")
            subprocess.run(command, check=True, capture_output=True, text=True, encoding='utf-8')
            print(f"Successfully created dataset {dataset_id}!")
        except subprocess.CalledProcessError as e:
            print("\nError creating dataset:")
            print("Stdout:", e.stdout)
            print("Stderr:", e.stderr)
            raise e

    def extract_multi_video(self, batch_size: int, width: int = 640, height: int = 480):
        """
        Extracts frames from multiple videos in batches and uploads them to Kaggle.
        """
        self.width = width
        self.height = height
        list_video = self.get_video_files()
        if not list_video:
            print("No videos found to process.")
            return

        num_batches = math.ceil(len(list_video) / batch_size)
        print(f"Total videos will be split into {num_batches} batches, with up to {batch_size} videos per batch.")

        for i in range(num_batches):
            batch_suffix = f"-batch-{i+1:03d}"
            dataset_name = self.base_dataset_name + batch_suffix
            print(f"\n{'='*20} Starting Batch {i + 1}/{num_batches} ({dataset_name}) {'='*20}")

            start_index = i * batch_size
            end_index = start_index + batch_size
            current_batch_videos = list_video[start_index:end_index]
            
            # Delete the upload directory before each batch to ensure it's empty
            shutil.rmtree(self.upload_dir, ignore_errors=True)
            self.upload_dir.mkdir(exist_ok=True)

            with Pool(cpu_count()) as p:
                iterator = p.imap_unordered(self._process_single_video, current_batch_videos)
                for processed_folder_path in tqdm(iterator, total=len(current_batch_videos), desc=f"Processing Batch {i+1}/{num_batches}"):
                    # Archive the frame directory into a zip file in the upload directory
                    zip_filename_base = self.upload_dir / processed_folder_path.name
                    shutil.make_archive(str(zip_filename_base), 'zip', str(processed_folder_path))
                    # Delete the frame directory after archiving
                    shutil.rmtree(processed_folder_path)

            batch_size_gb = self._get_dir_size_in_gb(self.upload_dir)
            print(f"Finished processing Batch {i + 1}. Total zip files size: {batch_size_gb:.4f} GB.")
            if batch_size_gb == 0:
                print("Warning: Batch is empty, no data to upload. Skipping this batch.")
                continue

            upload_message = f"Creating dataset {dataset_name} from Batch {i+1}/{num_batches}."
            self._create_new_dataset(dataset_name, upload_message)

        print(f"\n{'='*20} ALL BATCHES COMPLETED {'='*20}")

In [ ]:
import json, os
from pathlib import Path

INPUT_VIDEOS_PATH = '/kaggle/input/video-sokhao-l1-b1'
OUTPUT_PATH = '/kaggle/working/temp'
KAGGLE_API_CONFIG = {
    "username": "baoquocbao0929",
    "key": "db02e9489be1e4586bbd84b44e011069"
}
DATASET_NAME = "aic-2024-frames-part-1"
BATCH_SIZE=95
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# option A: run with start and end index
progress_config = {
    "start_index": 0,
    "end_index": 1000,
}

# option B: run all
# progress_config = {}

try:
    # Khởi tạo lớp FrameExtractor với cấu hình tiến trình đã chọn
    extractor = FrameExtractor(
        input_path=INPUT_VIDEOS_PATH,
        output_path=OUTPUT_PATH,
        kaggle_api=KAGGLE_API_CONFIG,
        base_dataset_name=DATASET_NAME,
        progress_info=progress_config  # Truyền cấu hình vào đây
    )
    
    # Bắt đầu quá trình trích xuất
    extractor.extract_multi_video(
        batch_size=BATCH_SIZE,
        width=FRAME_WIDTH, 
        height=FRAME_HEIGHT
    )
    
except Exception as e:
    print(f"Một lỗi không mong muốn đã xảy ra: {e}")

Total videos will be split into 2 batches, with up to 95 videos per batch.

==================== Starting Batch 1/2 (aic-2024-frames-part-1-batch-001) ====================


Processing Batch 1/2:   0%|          | 0/95 [00:00<?, ?it/s]